# 🧠 Chain-of-Thought Prompting

**Improve reasoning with step-by-step thinking**

---

## 📋 Overview

**What you'll learn:**
- What is Chain-of-Thought (CoT)
- Zero-shot CoT vs Few-shot CoT
- When CoT improves accuracy
- Self-consistency decoding
- Production CoT patterns

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
from openai import OpenAI
import os
from typing import List, Dict

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✅ Setup complete")

## 🤔 What is Chain-of-Thought?

**Chain-of-Thought (CoT)** = Ask LLM to show its reasoning steps

### Without CoT:
```
Q: If John has 3 apples and buys 2 more, then gives away 1, how many does he have?
A: 4
```
✅ Correct, but **no reasoning shown**

### With CoT:
```
Q: If John has 3 apples and buys 2 more, then gives away 1, how many does he have?
A: Let's solve this step by step:
1) John starts with: 3 apples
2) He buys 2 more: 3 + 2 = 5 apples
3) He gives away 1: 5 - 1 = 4 apples
Therefore, John has 4 apples.
```
✅ Correct, **and we can verify the reasoning**

### Why CoT Works:
- 🎯 **Better accuracy** on complex tasks
- 🔍 **Transparent reasoning** (explainability)
- 🐛 **Easier debugging** (see where it went wrong)
- 📚 **Works with smaller models** (GPT-3.5 vs GPT-4)

## 🔄 Zero-Shot CoT (Simplest!)

In [ ]:
def zero_shot_cot(question: str) -> str:
    """Zero-shot CoT: Just add 'Let's think step by step'"""
    
    prompt = f"{question}\n\nLet's think step by step:"
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=300
    )
    
    return response.choices[0].message.content

# Test with a math problem
question = """A restaurant has 23 tables. Each table can seat 4 people.
If the restaurant is 75% full, how many people are dining?"""

print("❓ Question:")
print(question)
print("\n🧠 Zero-Shot CoT Answer:")
print(zero_shot_cot(question))

## 📚 Few-Shot CoT (With Examples)

In [ ]:
def few_shot_cot(question: str, examples: List[Dict]) -> str:
    """Few-shot CoT with reasoning examples."""
    
    # Build prompt with examples
    prompt_parts = []
    
    for i, example in enumerate(examples, 1):
        prompt_parts.append(f"Example {i}:")
        prompt_parts.append(f"Q: {example['question']}")
        prompt_parts.append(f"A: {example['reasoning']}")
        prompt_parts.append("")
    
    prompt_parts.append("Now you try:")
    prompt_parts.append(f"Q: {question}")
    prompt_parts.append("A: Let's solve this step by step:")
    
    prompt = "\n".join(prompt_parts)
    
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=300
    )
    
    return response.choices[0].message.content

# Math examples with reasoning
math_examples = [
    {
        "question": "Roger has 5 tennis balls. He buys 2 more cans of 3 tennis balls each. How many does he have?",
        "reasoning": """Let's solve step by step:
1) Roger starts with: 5 balls
2) He buys 2 cans with 3 balls each: 2 × 3 = 6 balls
3) Total: 5 + 6 = 11 balls
Therefore, Roger has 11 tennis balls."""
    },
    {
        "question": "The cafeteria had 23 apples. If they used 20 to make lunch and bought 6 more, how many apples do they have?",
        "reasoning": """Let's solve step by step:
1) Started with: 23 apples
2) Used for lunch: 23 - 20 = 3 apples left
3) Bought 6 more: 3 + 6 = 9 apples
Therefore, the cafeteria has 9 apples."""
    }
]

# Test with new question
new_question = "A baker made 15 cakes. She sold 9 and then made 12 more. How many cakes does she have now?"

print("❓ Question:")
print(new_question)
print("\n🧠 Few-Shot CoT Answer:")
print(few_shot_cot(new_question, math_examples))

## 🎯 Comparing: Direct vs CoT

In [ ]:
def direct_answer(question: str) -> str:
    """Get answer without CoT."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": question}],
        temperature=0,
        max_tokens=100
    )
    return response.choices[0].message.content

# Complex logic problem
problem = """In a race, you passed the person in second place.
What place are you in now?"""

print("❓ Question:")
print(problem)

print("\n📌 Direct Answer (no reasoning):")
print(direct_answer(problem))

print("\n🧠 CoT Answer (with reasoning):")
print(zero_shot_cot(problem))

print("\n💡 Notice: CoT shows the logic clearly!")

## 🔁 Self-Consistency (Multiple Paths)

In [ ]:
from collections import Counter
import re

def extract_final_answer(text: str) -> str:
    """Extract the final numerical answer from reasoning."""
    # Look for patterns like "Therefore, X" or "The answer is X"
    patterns = [
        r'Therefore,?\s+(?:the answer is\s+)?(\d+)',
        r'(?:The )?answer is\s+(\d+)',
        r'has\s+(\d+)\s+\w+\.$',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return match.group(1)
    
    # Fallback: find last number
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else "unknown"

def self_consistency(question: str, n: int = 5) -> Dict:
    """Generate multiple reasoning paths and pick most common answer."""
    
    prompt = f"{question}\n\nLet's think step by step:"
    
    responses = []
    answers = []
    
    print(f"🔄 Generating {n} reasoning paths...\n")
    
    for i in range(n):
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,  # Higher temp for diversity
            max_tokens=300
        )
        
        reasoning = response.choices[0].message.content
        answer = extract_final_answer(reasoning)
        
        responses.append(reasoning)
        answers.append(answer)
        
        print(f"  Path {i+1}: Answer = {answer}")
    
    # Find most common answer
    answer_counts = Counter(answers)
    most_common_answer, count = answer_counts.most_common(1)[0]
    
    print(f"\n📊 Answer distribution: {dict(answer_counts)}")
    print(f"✅ Most common answer: {most_common_answer} ({count}/{n} paths)")
    
    return {
        'final_answer': most_common_answer,
        'confidence': count / n,
        'all_answers': answers,
        'reasoning_paths': responses
    }

# Test self-consistency
question = """A store had 20 oranges. They sold some and had 15 left.
Then they got a shipment of 30 more oranges. How many oranges do they have now?"""

result = self_consistency(question, n=5)

print(f"\n🎯 Final answer with {result['confidence']*100:.0f}% confidence: {result['final_answer']}")

## 🏗️ Production CoT Template

In [ ]:
class ChainOfThoughtEngine:
    """Production-ready CoT engine."""
    
    def __init__(self, model: str = "gpt-3.5-turbo"):
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        self.model = model
    
    def zero_shot(self, question: str, instruction: str = None) -> Dict:
        """Zero-shot CoT with custom instruction."""
        instruction = instruction or "Let's think step by step:"
        prompt = f"{question}\n\n{instruction}"
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=500
        )
        
        reasoning = response.choices[0].message.content
        
        return {
            'question': question,
            'reasoning': reasoning,
            'tokens': response.usage.total_tokens,
            'method': 'zero_shot_cot'
        }
    
    def few_shot(self, question: str, examples: List[Dict]) -> Dict:
        """Few-shot CoT."""
        prompt_parts = []
        
        for i, ex in enumerate(examples, 1):
            prompt_parts.append(f"Example {i}:")
            prompt_parts.append(f"Q: {ex['question']}")
            prompt_parts.append(f"A: {ex['reasoning']}")
            prompt_parts.append("")
        
        prompt_parts.append("Now you try:")
        prompt_parts.append(f"Q: {question}")
        prompt_parts.append("A:")
        
        prompt = "\n".join(prompt_parts)
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=500
        )
        
        return {
            'question': question,
            'reasoning': response.choices[0].message.content,
            'examples_used': len(examples),
            'tokens': response.usage.total_tokens,
            'method': 'few_shot_cot'
        }
    
    def with_self_consistency(
        self,
        question: str,
        n_paths: int = 5
    ) -> Dict:
        """CoT with self-consistency."""
        prompt = f"{question}\n\nLet's solve this step by step:"
        
        responses = []
        for _ in range(n_paths):
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=300
            )
            responses.append(response.choices[0].message.content)
        
        # Extract answers and find most common
        answers = [extract_final_answer(r) for r in responses]
        answer_counts = Counter(answers)
        final_answer, count = answer_counts.most_common(1)[0]
        
        return {
            'question': question,
            'final_answer': final_answer,
            'confidence': count / n_paths,
            'reasoning_paths': responses,
            'all_answers': answers,
            'method': 'self_consistency'
        }

# Test production engine
engine = ChainOfThoughtEngine()

question = "If a train travels 120 miles in 2 hours, what is its speed in miles per hour?"

print("🧪 Testing CoT Engine\n")
print("="*50)

# Zero-shot
result = engine.zero_shot(question)
print("\n📌 Zero-Shot CoT:")
print(result['reasoning'])
print(f"Tokens: {result['tokens']}")

## 📊 When to Use CoT

### ✅ Use CoT for:
- **Math problems** (calculations, word problems)
- **Logic puzzles** (reasoning required)
- **Multi-step tasks** (planning, debugging)
- **Code explanation** (step-by-step logic)
- **Complex analysis** (requires reasoning)

### ❌ Don't use CoT for:
- **Simple lookups** ("What is the capital of France?")
- **Classifications** (sentiment, topic)
- **Short responses** (yes/no questions)
- **Factual recall** ("Who wrote X?")

### 💰 Cost Considerations:
```python
Direct:        100 tokens  → $0.00005
Zero-shot CoT: 300 tokens  → $0.00015 (3x cost)
Few-shot CoT:  600 tokens  → $0.00030 (6x cost)
```

**Use CoT only when accuracy > cost**

## ✅ Summary

### Key Concepts:

1. **🧠 Chain-of-Thought**
   - Show reasoning steps
   - Improves accuracy on complex tasks
   - Better explainability

2. **🔄 Zero-Shot CoT**
   - Simplest: Just add "Let's think step by step"
   - Works surprisingly well
   - No examples needed

3. **📚 Few-Shot CoT**
   - Provide reasoning examples
   - Better for specific formats
   - Higher token cost

4. **🔁 Self-Consistency**
   - Generate multiple paths
   - Pick most common answer
   - Higher confidence

### CoT Prompts Library:

```python
# Math
"Let's solve this step by step:"

# Logic
"Let's think through this carefully:"

# Debugging
"Let's debug this step by step:
1) What is the code trying to do?
2) What is the error?
3) Where is the bug?
4) How to fix it?"

# Analysis
"Let's analyze this systematically:
1) What do we know?
2) What can we infer?
3) What is the conclusion?"
```

### Best Practices:

1. **Temperature**
   - Zero-shot: temp=0 (deterministic)
   - Self-consistency: temp=0.7 (diversity)

2. **When to use which:**
   - Simple task → No CoT
   - Complex task → Zero-shot CoT
   - Need high accuracy → Self-consistency
   - Specific format → Few-shot CoT

3. **Cost optimization:**
   - Use on complex tasks only
   - Cache common reasoning paths
   - Start with zero-shot before few-shot

### Performance Gains:
```
Task: Math word problems (GSM8K dataset)
Direct answer:        40% accuracy
Zero-shot CoT:        60% accuracy (+20%)
Few-shot CoT:         70% accuracy (+30%)
Self-consistency:     75% accuracy (+35%)
```

### Next: `03_prompt_engineering/06_few_shot_learning.ipynb`